In [6]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool, ToolRuntime
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

<div style="font: bold 40px sans-serif;">LangChain Middleware 中间件</div>

<br/>

中间件是一种控制Agent内部运行过程的技术，它在智能体运行的各个过程中预留钩子（hook），方便我们嵌入自定义操作。

# Middleware核心功能

- **拦截和修改请求** - 在模型调用前后对输入输出进行处理
- **实现PII脱敏** - 自动检测和脱敏敏感个人信息
- **会话摘要管理** - 当对话过长时自动压缩历史消息
- **人工审核机制** - 在执行危险操作前等待人工确认
- **动态模型选择** - 根据运行时条件选择不同的模型
- **自定义状态管理** - 扩展Agent状态以跟踪额外信息

Agent的默认执行流程和中间件嵌入钩子的对比如图：

<div style="display:flex; justify-content:center; align-items:center; gap:20px;">
<img src="./resources/agents-loop.png" style="max-width:400px; height:auto;" alt="Agents Loop">
<img src="./resources/hooks.png" style="max-width:100%; height:auto;" alt="Hooks">
</div>

如上图所示，LangChain的Middleware基于钩子（Hooks）机制，主要有以下几种类型：

| 钩子类型 | 说明 |
|---------|------|
| `before_model` | 在模型调用之前执行，可修改输入 |
| `after_model` | 在模型调用之后执行，可处理输出 |
| `wrap_model_call` | 包装整个模型调用过程，可完全控制执行流程 |
| `pre_model_hook` | LangChain v0中等效于before_model |
| `post_model_hook` | LangChain v0中等效于after_model |

# 2. 预置中间件（Built-in Middleware）

LangChain提供了多种预置中间件，可以直接使用。

## 2.1 PIIMiddleware - 个人信息脱敏

自动检测并脱敏消息中的个人身份信息（PII），如邮箱、电话号码、身份证号等。
其中，PII的处理策略有四种：
- `'block'` - 抛出异常
- `'redact'` - 用 [REDACTED_{PII_TYPE}] 来替代
- `'mask'` - 关键信息采用**掩码 (例如., `****-****-****-1234`)
- `'hash'` - 用哈希值来替换


In [6]:
from langchain.agents.middleware import PIIMiddleware

# 也可以添加多个PII中间件来检测不同类型的敏感信息
pii_middleware_email = PIIMiddleware(
    "email",
    strategy="redact",  # mask
    apply_to_input=False,
    apply_to_output=True
)

pii_middleware_phone = PIIMiddleware(
    "phone_number",
    # 使用自定义正则表达式
    detector=r"(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?)?\d{3,4}[\s.-]?\d{4}",
    strategy="block",
    apply_to_input=True
)

print("PIIMiddleware 配置完成")
print(f"- 邮箱脱敏策略: {pii_middleware_email.strategy}")
print(f"- 电话号码策略: {pii_middleware_phone.strategy}")

PIIMiddleware 配置完成
- 邮箱脱敏策略: redact
- 电话号码策略: block


In [7]:
agent_with_builtin_middleware = create_agent(
    model="deepseek-v4-flash",
    middleware=[pii_middleware_email, pii_middleware_phone]
)

In [5]:
response = agent_with_builtin_middleware.invoke({
    "messages": [
        HumanMessage(
            "我的邮箱是: huge@itcast.cn,我的电话是13698023405,你要记住这些信息，后续要用到。如果记住了请确认一遍。")
    ]
})
for m in response['messages']:
    m.pretty_print()

PIIDetectionError: Detected 1 instance(s) of phone_number in text content

## 2.2 ModelFallbackMiddleware

可以在创建智能体时设置多个模型，如果第一个模型调用失败，会自动调用下一个模型。


In [26]:
from langchain.agents.middleware import ModelFallbackMiddleware

model_fallback_middleware = ModelFallbackMiddleware(
    "gpt-4o-mini",  # 默认模型
    "deepseek-v4-flash"  #
)

agent_with_model_fallback = create_agent(
    model="deepseek-v4-flash",
    middleware=[model_fallback_middleware]
)

In [27]:
response = agent_with_builtin_middleware.invoke({
    "messages": [HumanMessage("你好")]
})
for m in response['messages']:
    m.pretty_print()

================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！很高兴见到你！😊 我是DeepSeek，由深度求索公司创造的AI助手。无论你有什么问题、需要什么帮助，或者只是想聊聊天，我都很乐意为你提供支持！

我可以帮你处理各种任务，比如：
- 回答问题和解释概念
- 协助写作和翻译
- 分析和处理文档
- 编程和技术支持
- 创意思考和头脑风暴

有什么我可以为你做的吗？请随时告诉我你的需求！✨


## 2.3 HumanInTheLoopMiddleware - 人工审核

在执行特定工具调用前暂停，等待人工确认。可用于控制敏感操作如发送邮件、转账等。

In [14]:
from langchain.agents.middleware import HumanInTheLoopMiddleware


@tool
def transfer_money(amount: int, to: str):
    """向指定账户转账
    :arg
    amount: 金额
    to: 收款人
    """
    return f"已向账户{to}转账{amount}元"


# 创建人工审核中间件
human_in_loop_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "transfer_money": {
            "description": "请确认转账操作",
            "allowed_decisions": ["approve", "reject", "edit"]
        }
    }
)


In [4]:
agent_with_hitl = create_agent(
    model="deepseek-v4-flash",
    tools=[transfer_money],
    middleware=[human_in_loop_middleware],
    checkpointer=InMemorySaver(),
    system_prompt="你是一个账户管理助手，你可以调用工具帮助用户转账，用户提出明确需求后，不要重复向用户确认信息。"
)

In [46]:
config = {"configurable": {"thread_id": "3"}}
response = agent_with_hitl.invoke(
    {"messages": [HumanMessage("帮我转2000元给王小明(6123008415124395223)")]},
    config=config
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我转2000元给王小明(6123008415124395223)
================================== Ai Message ==================================

我来帮您转账2000元给王小明。
Tool Calls:
  transfer_money (call_00_ttQGhF2bM9l4ZctZWtc9pFeU)
 Call ID: call_00_ttQGhF2bM9l4ZctZWtc9pFeU
  Args:
    amount: 2000
    to: 6123008415124395223


In [47]:
print(response)

{'messages': [HumanMessage(content='帮我转2000元给王小明(6123008415124395223)', additional_kwargs={}, response_metadata={}, id='18b33155-fec2-46fb-8bb4-2c0cd414a173'), AIMessage(content='我来帮您转账2000元给王小明。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 365, 'total_tokens': 442, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 365}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410', 'id': '894be682-141b-4dbe-9370-80f858c05982', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019daf31-b706-71d1-8340-b42f16552abe-0', tool_calls=[{'name': 'transfer_money', 'args': {'amount': 2000, 'to': '6123008415124395223'}, 'id': 'call_00_ttQGhF2bM9l4ZctZWtc9pFeU', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'inpu

In [48]:
from pprint import pprint

pprint(response['__interrupt__'][0].value)

{'action_requests': [{'args': {'amount': 2000, 'to': '6123008415124395223'},
                      'description': '请确认转账操作',
                      'name': 'transfer_money'}],
 'review_configs': [{'action_name': 'transfer_money',
                     'allowed_decisions': ['approve', 'reject', 'edit']}]}


### approve



In [22]:
from langgraph.types import Command

response = agent_with_hitl.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config  # Same thread ID to resume the paused conversation
)

print(response)

{'messages': [HumanMessage(content='帮我转2000元给王小明(6123008415124395223)', additional_kwargs={}, response_metadata={}, id='157ef049-eed6-4abb-a03e-a92f2335fd09'), AIMessage(content='我来帮您转账2000元给王小明。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 352, 'total_tokens': 432, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 320}, 'prompt_cache_hit_tokens': 320, 'prompt_cache_miss_tokens': 32}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410', 'id': 'bc35e916-05a7-4844-b2ca-a658e01035e5', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019daf1c-d712-7721-9c7e-7721cddeb785-0', tool_calls=[{'name': 'transfer_money', 'args': {'amount': 2000, 'to': '王小明(6123008415124395223)'}, 'id': 'call_00_TiBI1Cfgz5R5i8bnIhbsFDuW', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadat

### Reject

In [40]:
response = agent_with_hitl.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "算了，暂时不转了。"
                }
            ]
        }
    ),
    config=config  # Same thread ID to resume the paused conversation
)

print(response)

{'messages': [HumanMessage(content='帮我转2000元给王小明(6123008415124395223)', additional_kwargs={}, response_metadata={}, id='157ef049-eed6-4abb-a03e-a92f2335fd09'), AIMessage(content='我来帮您转账2000元给王小明。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 352, 'total_tokens': 432, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 320}, 'prompt_cache_hit_tokens': 320, 'prompt_cache_miss_tokens': 32}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410', 'id': 'bc35e916-05a7-4844-b2ca-a658e01035e5', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019daf1c-d712-7721-9c7e-7721cddeb785-0', tool_calls=[{'name': 'transfer_money', 'args': {'amount': 2000, 'to': '王小明(6123008415124395223)'}, 'id': 'call_00_TiBI1Cfgz5R5i8bnIhbsFDuW', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadat

### edit


In [49]:
response = agent_with_hitl.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "transfer_money",
                        # Arguments to pass to the tool.
                        "args": {"amount": 1000, "to": "王小明"},
                    }
                }
            ]
        }

    config=config  # Same thread ID to resume the paused conversation
)

pprint(response)

{'messages': [HumanMessage(content='帮我转2000元给王小明(6123008415124395223)', additional_kwargs={}, response_metadata={}, id='18b33155-fec2-46fb-8bb4-2c0cd414a173'),
              AIMessage(content='我来帮您转账2000元给王小明。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 365, 'total_tokens': 442, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 365}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410', 'id': '894be682-141b-4dbe-9370-80f858c05982', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019daf31-b706-71d1-8340-b42f16552abe-0', tool_calls=[{'type': 'tool_call', 'name': 'transfer_money', 'args': {'amount': 1000, 'to': '王小明'}, 'id': 'call_00_ttQGhF2bM9l4ZctZWtc9pFeU'}], invalid_tool_calls=[], usage_metadata={'input_

# 3. 自定义中间件

除了使用预置中间件，我们还可以利用LangChain提供的hook创建完全自定义的中间件。

根据hook的种类，中间件可以分为两类：
- Node-style hooks：在具体某个节点执行的中间件，包含：
  - before_agent
  - before_model
  - after_model
  - after_agent
- Wrap-style hooks：环绕model或tool调用的中间件，包括：
  - wrap_model_call
  - wrap_tool_call

为了便于开发中间件，LangChain为每一种hook都提供了装饰器，我们只有定义函数并使用装饰器标记即可快速开发中间件。


## 3.1 node-style装饰器

例如，我们要实现模型调用计数功能，可以在每次调用模型后(after_model)记录模型调用次数


In [8]:
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from typing import NotRequired, Any
from langchain.agents.middleware import after_model


# 1.定义自定义state，记录模型调用次数
class CustomAgentState(AgentState):
    """扩展Agent状态，添加自定义字段"""
    model_call_count: NotRequired[int]  # 模型调用次数


# 2.定义中间件
@after_model(state_schema=CustomAgentState)
def increment_counter(state: CustomAgentState, runtime: Runtime) -> dict[str, Any]:
    """使用装饰器的after_model钩子 - 增加调用计数"""
    current_count = state.get("model_call_count", 0)
    return {"model_call_count": current_count + 1}

In [9]:
# 创建智能体，设置middleware
agent = create_agent(
    model="deepseek-v4-flash",
    middleware=[increment_counter],
    checkpointer=InMemorySaver()
)
config = {"configurable": {"thread_id": "1"}}

# 调用智能体，并初始化state
response = agent.invoke({
    "messages": [HumanMessage("Hello")],
    "model_call_count": 0,
}, config)

print(response)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='f333c2ff-d50b-452c-941b-adb1c69812f6'), AIMessage(content='你好！有什么可以帮你的吗？😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 5, 'total_tokens': 15, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': 'fc0a7204-aafb-44f2-89a0-c516096134b1', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dbe9e-f818-7b82-86ab-dd39577897b2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 10, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})], 'model_call_count': 1}


In [10]:
# 调用智能体，并初始化state
response = agent.invoke({
    "messages": [HumanMessage("tell me a joke")]
}, config)

print(response)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='f333c2ff-d50b-452c-941b-adb1c69812f6'), AIMessage(content='你好！有什么可以帮你的吗？😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 5, 'total_tokens': 15, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': 'fc0a7204-aafb-44f2-89a0-c516096134b1', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dbe9e-f818-7b82-86ab-dd39577897b2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 10, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}), HumanMessage(content='tell me a joke', additional_kwargs={}, respo

## 3.2.wrap-style装饰器

例如，我们来定义一个可以在调用模型时失败重试的中间件，最大重试次数为3次


In [10]:
from langchain.agents.middleware import (
    wrap_model_call,
    ModelRequest,
    ModelResponse,
)
from typing import Any, Callable


@wrap_model_call
def retry_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request)
        except Exception as e:
            print(f"Retry {attempt + 1}/3 after error: {e}")
            if attempt == 2:
                raise
    return None


agent = create_agent(
    model="deepseek-v4-flash",
    middleware=[retry_model]
)

In [12]:
config = {"configurable": {"thread_id": "1"}}

# 调用智能体，并初始化state
response = agent.invoke(
    {"messages": [HumanMessage("Hello")]},
    config,
)

print(response)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='b15f92dd-c56f-492a-93a7-c80699ecf541'), AIMessage(content='你好！有什么可以帮你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 5, 'total_tokens': 13, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': '283f549b-671c-4e38-882c-3e663b521b11', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dbe9f-71e5-7143-8e31-817da54159cf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 8, 'total_tokens': 13, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})]}


## 3.2 使用类方式创建中间件

对于需要同时用到多个hooks的更复杂的中间件逻辑，建议使用类方式继承AgentMiddleware。

In [13]:
from langchain.agents.middleware import AgentMiddleware
from langchain.agents.middleware.types import ModelCallResult, ToolCallRequest
from langgraph.types import Command


class LoggingMiddleware(AgentMiddleware):

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelCallResult:
        try:
            print(f"\n=======About to call model with {len(request.messages)} messages=======")
            return handler(request)
        except Exception as e:
            print(f"\n=======[错误]: {str(e)}=======")
            return AIMessage("调用模型失败，请重试~")

    def wrap_tool_call(
        self,
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage | Command],
    ) -> ToolMessage | Command:
        print(f"\n=======调用工具: {request.tool_call['name']}=======")
        print(f"\n=======参数: {request.tool_call['args']}=======")
        try:
            result = handler(request)
            print("\n=======工具调用成功！=======")
            return result
        except Exception as e:
            print(f"\n=======工具调用失败: {e}=======")
            raise


@tool
def get_weather(location: str):
    """查询指定城市的天气信息"""
    return f"Current weather in {location} is sunny, 25℃."


agent = create_agent(
    model="deepseek-v4-flash",
    middleware=[LoggingMiddleware()],
    tools=[get_weather],
)

In [14]:
for chunk, metadata in agent.stream(
    {"messages": [HumanMessage("杭州今天天气如何？")]},
    stream_mode="messages"
):
    if chunk and chunk.content:
        print(chunk.content, end="", flush=True)


=======About to call model with 1 messages=======
好的，我来查询一下杭州今天的天气情况。
=======调用工具: get_weather=======

=======参数: {'location': '杭州'}=======

=======工具调用成功！=======
Current weather in 杭州 is sunny, 25℃.
=======About to call model with 3 messages=======
杭州今天天气晴朗，气温 **25℃**，非常适合外出活动哦！🌞 注意适当防晒～

# 4. 高级用法

## 4.1 动态选择模型

在wrap_model_call这个hook中，我们可以动态修改任意的request参数，包括：
- model
- tool
- system_prompt
- ...

例如，我们定义一个用来动态切换model的中间件，在每次模型调用前判断是采用推理模型还是普通模型。


In [15]:
# 使用 @wrap_model_call 装饰器创建中间件
# 这种方式适合简单的请求修改
from langchain.agents.middleware import wrap_model_call
from collections.abc import Callable
from pydantic.dataclasses import dataclass

# 初始化不同的模型
reasoning_model = init_chat_model(model="deepseek-reasoner")
chat_model = init_chat_model(model="deepseek-v4-flash")


# 定义一个context，记录用户是否使用推理模型
@dataclass
class UserContext:
    reasoning: bool = False


@wrap_model_call
def dynamic_model_selector(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """
    根据context中的reasoning来选择模型：
    - True: 选择推理模型
    - False: 选择普通对话模型
    """
    # 获取上下文参数
    is_reasoning = getattr(request.runtime.context, 'reasoning', False)

    # 根据消息数量选择模型
    selected_model = reasoning_model if is_reasoning else chat_model
    print(f"选择模型: {selected_model.model_name} (reasoning={is_reasoning})")

    # 覆盖request中的model参数
    modified_request = request.override(model=selected_model)
    return handler(modified_request)

In [16]:
# 创建智能体，设置middleware
agent = create_agent(
    model="deepseek-v4-flash",
    middleware=[dynamic_model_selector],
    checkpointer=InMemorySaver(),
    context_schema=UserContext
)
config = {"configurable": {"thread_id": "1"}}

# 调用智能体，并初始化state
response = agent.invoke(
    {"messages": [HumanMessage("Hello")]},
    config,
    context=UserContext(reasoning=True)
)

print(response)

选择模型: deepseek-reasoner (reasoning=True)
{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='3a766498-5a86-4fc2-ac1b-f394a2158368'), AIMessage(content='你好！很高兴见到你。😊\n\n有什么我可以帮你的吗？无论是想聊聊天、解答问题，还是帮你处理文件、查询资料，我都在这里！', additional_kwargs={'refusal': None, 'reasoning_content': '好的，用户只发了一个“Hello”，这是一个非常简单的问候。问题本身没有复杂指令，不需要深度推理。用户可能只是想打个招呼，或者测试我是否在回应。深层需求可能是开启对话，或者看看我的反应是否友好自然。\n\n我需要用同样友好、热情的语气回应，同时主动提供一个开放式的邀请，让用户有机会提出进一步的问题或说明意图。想到了用“你好！很高兴见到你。”开头，直接呼应问候，然后加上“有什么我可以帮你的吗？”来引导对话继续，最后用表情符号增加亲和力。'}, response_metadata={'token_usage': {'completion_tokens': 150, 'prompt_tokens': 5, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 114, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerpri

## 4.2 条件跳转

使用`jump_to`实现在特定条件下跳过模型调用。
例如，我们设定一个快速应答的中间件，当用户提问符合预设词时，我们跳过模型调用，直接返回结果。


In [33]:
from langgraph.runtime import Runtime
from typing import NotRequired, Any
from langchain.agents.middleware import AgentMiddleware, hook_config, AgentState


# 1.定义自定义state，记录模型调用次数
class CustomAgentState(AgentState):
    """扩展Agent状态，添加自定义字段"""
    model_call_count: NotRequired[int]  # 模型调用次数


# 2.定义中间件
class ModelCallLimitMiddleware(AgentMiddleware[CustomAgentState]):

    def __init__(self, max_limit: int = 10):
        super().__init__()
        self.max_limit = max_limit

    # after_model hook，模型调用次数计数
    def after_model(self, state: CustomAgentState, runtime: Runtime) -> dict[str, Any] | None:
        current_count = state.get("model_call_count", 0)
        return {"model_call_count": current_count + 1}

    # before_model hook，校验模型调用次数是否超限
    @hook_config(can_jump_to=["end"])
    def before_model(self, state: CustomAgentState, runtime: Runtime) -> dict[str, Any] | None:
        # 获取模型调用次数
        current_count = state.get("model_call_count", 0) + 1
        # 判断是否超限
        print(f"当前模型调用次数：{current_count}/{self.max_limit}")
        if current_count > self.max_limit:
            return {
                "jump_to": "end",
                "messages": AIMessage(f"模型调用次数超过最大限制：{self.max_limit}！")
            }
        return None

# @before_model(can_jump_to=["end"])
# def check_call_limit(state: CustomAgentState, runtime: Runtime) -> dict[str, Any] | None:
#     pass

In [34]:
# 创建智能体，设置middleware
agent = create_agent(
    model="deepseek-v4-flash",
    middleware=[ModelCallLimitMiddleware(max_limit=2)],
    checkpointer=InMemorySaver(),
    state_schema=CustomAgentState
)
config = {"configurable": {"thread_id": "1"}}

# 调用智能体，并初始化state
response = agent.invoke({
    "messages": [HumanMessage("Hello")],
    "model_call_count": 0,
}, config)

print(response)

当前模型调用次数：1/2
{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='226591f8-aab8-4a1e-8ba4-683f8ef89cb5'), AIMessage(content='你好！😊 有什么可以帮你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 5, 'total_tokens': 16, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': 'fd927e59-83b9-479e-a000-9c4e903f6d3b', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dbeab-e26e-7383-aef1-2653193929b7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 11, 'total_tokens': 16, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})], 'model_call_count': 1}


In [35]:
# 调用智能体，并初始化state
response = agent.invoke({
    "messages": [HumanMessage("tell me a joke")]
}, config)

print(response)

当前模型调用次数：2/2
{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='226591f8-aab8-4a1e-8ba4-683f8ef89cb5'), AIMessage(content='你好！😊 有什么可以帮你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 5, 'total_tokens': 16, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': 'fd927e59-83b9-479e-a000-9c4e903f6d3b', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dbeab-e26e-7383-aef1-2653193929b7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 11, 'total_tokens': 16, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}), HumanMessage(content='tell me a joke', additional_kw

In [36]:
# 调用智能体，并初始化state
response = agent.invoke({
    "messages": [HumanMessage("tell me another joke")]
}, config)

print(response)

当前模型调用次数：3/2
{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='226591f8-aab8-4a1e-8ba4-683f8ef89cb5'), AIMessage(content='你好！😊 有什么可以帮你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 5, 'total_tokens': 16, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': 'fd927e59-83b9-479e-a000-9c4e903f6d3b', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dbeab-e26e-7383-aef1-2653193929b7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 11, 'total_tokens': 16, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}), HumanMessage(content='tell me a joke', additional_kw

---

**更多资源**:
- [LangChain官方文档](https://docs.langchain.com/oss/python/langchain/middleware/custom)
- [LangChain GitHub](https://github.com/langchain-ai/langchain)